In [1]:
text = """
<think> Okay, let's see. The user provided some text from two markdown files, characters.md and planets.md. They mention Yoda, his green lightsaber, training with Yoda on Dagobah, and details about Dagobah being a swamp planet where Yoda lived in exile after Order 66.
First, I need to understand what the user is asking. Since they included "Fonte" (which means "Source" in Portuguese), maybe they want me to combine the information into a coherent summary or answer a question based on these sources. But the user hasn't explicitly asked a question yet. The last message just ends with the provided text. However, looking at the pattern, maybe the user wants me to generate a response based on the given sources, similar to the previous examples where they provided context and then a question.

Wait, in the history, the user previously asked about Yoda's training and Dagobah. The assistant then provided a structured answer. Now, the user is giving more sources and maybe expects a similar structured response. Let me check the current input again.

The user provided two sources: characters.md and planets.md. The first source says "Seu sabre de luz é verde." (Your lightsaber is green) and talks about Yoda as a Jedi Master. The second part mentions training with Yoda on Dagobah. Then from planets.md, Dagobah is a swamp planet where Yoda lived in exile after Order 66.

The user might be asking to combine these facts into a coherent answer, perhaps about Yoda's role, his lightsaber color, his training location, and Dagobah's significance. They might want a summary that connects these elements.

I should structure the answer by first introducing Yoda, his lightsaber color, his role as a Jedi Master, his training with Luke on Dagobah, and describe Dagobah as a swamp planet where he was exiled after Order 66. Make sure to cite the sources correctly as in the examples. Also, check for any possible connections or additional context that might be implied but not stated, like Yoda's importance in the Jedi Order or the significance of Dagobah in the Star Wars saga.

Wait, in the provided text, the user wrote "seu treinamento com Yoda no planeta Dagobah." That translates to "your training with Yoda on the planet Dagobah." The use of "seu" might be a typo for "Luke's training," assuming the context is from the perspective of Luke Skywalker. But the user hasn't specified. However, given the context of Star Wars, it's safe to assume that refers to Luke's training with Yoda.

So, the answer should mention that Yoda trained Luke Skywalker on Dagobah, which is a swamp planet where Yoda lived in exile after the events of Order 66. Also, Yoda's green lightsaber is a notable trait, indicating his connection to the Force. Need to make sure the answer is concise, uses the sources provided, and presents the information clearly with proper citations. </think>

Com base nas fontes fornecidas, aqui está a síntese das informações:
"""

In [12]:
def extract_thinking(res: str):
    res = res.strip()

    start_think_pos = res.find('<think>')
    end_think_pos = res.find('</think>')
    if start_think_pos != -1 and end_think_pos != -1:
        res = res[start_think_pos:end_think_pos]
        return res + "</think>"
    else:
        return ""

In [13]:
s = extract_thinking(text)
s

'<think> Okay, let\'s see. The user provided some text from two markdown files, characters.md and planets.md. They mention Yoda, his green lightsaber, training with Yoda on Dagobah, and details about Dagobah being a swamp planet where Yoda lived in exile after Order 66.\nFirst, I need to understand what the user is asking. Since they included "Fonte" (which means "Source" in Portuguese), maybe they want me to combine the information into a coherent summary or answer a question based on these sources. But the user hasn\'t explicitly asked a question yet. The last message just ends with the provided text. However, looking at the pattern, maybe the user wants me to generate a response based on the given sources, similar to the previous examples where they provided context and then a question.\n\nWait, in the history, the user previously asked about Yoda\'s training and Dagobah. The assistant then provided a structured answer. Now, the user is giving more sources and maybe expects a simila

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings

__embedding_model__ = HuggingFaceEmbeddings(model_name="/home/jpmaida/.cache/huggingface/hub/models--BAAI--bge-m3/snapshots/5617a9f61b028005a4858fdac845db406aefb181")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [7]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.load_local("../database/index_faiss", __embedding_model__, allow_dangerous_deserialization=True)

In [8]:
print(type(vectorstore.index))

<class 'faiss.swigfaiss.IndexFlatL2'>


In [2]:
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

def generate_chunks_and_metadata(chunk_size=100) -> list:
    KNOWLEDGE_PATH = Path("../knowledge")
    docs = list(KNOWLEDGE_PATH.glob("*.md"))
    knowledge_as_text = []
    for d in docs:
        if d.is_file:
            content = d.read_text(encoding='utf-8')
            knowledge_as_text.append(content)
    
    # 20% for overlapping
    chunk_overlap = chunk_size * 0.2
    
    text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=chunk_size,
      chunk_overlap=chunk_overlap
    )

    chunks = []
    metadatas = []
    for index, k in enumerate(knowledge_as_text):
        splits = text_splitter.split_text(k)
        chunks.extend(splits)
        doc = docs[index]
        for s in splits:
            metadatas.append({
                "source": doc.name
            })
    
    return chunks, metadatas

In [30]:
import numpy as np
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

chunks, metadatas = generate_chunks_and_metadata()
embeddings = np.array(object=__embedding_model__.embed_documents(chunks),
                                dtype=np.float32)
metric = "L2"

if metric == "L2":
    index = faiss.IndexFlatL2(embeddings.shape[1])
elif metric == "IP":
    index = faiss.IndexFlatIP(embeddings.shape[1])

index.add(embeddings)

vectorstore = FAISS(
    embedding_function=__embedding_model__,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

print(type(vectorstore.index))

<class 'faiss.swigfaiss.IndexFlatL2'>


In [31]:
chunks, metadatas = generate_chunks_and_metadata()
vectorstore = FAISS.from_texts(
            texts=chunks, 
            embedding=__embedding_model__, 
            metadatas=metadatas
        )


In [35]:
dimension = vectorstore.index.d
print(dimension)
print(vectorstore.index)

1024
<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x7f9ec15e54f0> >


In [3]:
import sys
# Replace with the actual folder path containing your .py file
sys.path.append("..") 

from src.vector_database import *

In [ ]:
chunks, metadatas = generate_chunks_and_metadata()
db = VectorDatabase()
vectorstore = db.build_vectorstore(chunks=chunks, metadatas=metadatas)
question = "O que é Dagobah?"
query_embedding = __embedding_model__.embed_query(question)
results = db.search(vectorstore, query_embedding=query_embedding)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [10]:
results

[{'score': 0.8445559740066528,
  'document': Document(metadata={'source': 'planets.md'}, page_content='---\n\n## Dagobah\n\nPlaneta pantanoso.\n\nYoda viveu exilado em Dagobah após a Ordem 66.'),
  'chunk': '---\n\n## Dagobah\n\nPlaneta pantanoso.\n\nYoda viveu exilado em Dagobah após a Ordem 66.',
  'metadata': {'source': 'planets.md'}},
 {'score': 0.8985930681228638,
  'document': Document(metadata={'source': 'characters.md'}, page_content='seu treinamento com Yoda no planeta Dagobah.'),
  'chunk': 'seu treinamento com Yoda no planeta Dagobah.',
  'metadata': {'source': 'characters.md'}},
 {'score': 0.902370810508728,
  'document': Document(metadata={'source': 'planets.md'}, page_content='É o planeta natal de Luke Skywalker.\n\nTambém foi onde Anakin Skywalker nasceu.\n\n---\n\n## Dagobah'),
  'chunk': 'É o planeta natal de Luke Skywalker.\n\nTambém foi onde Anakin Skywalker nasceu.\n\n---\n\n## Dagobah',
  'metadata': {'source': 'planets.md'}}]